# Equation to speech

How do you say `(a+b)^2` out loud in Hindi?

Hand `(a+b)^2` to any text-to-speech engine and you get the *characters* read out, not the
*maths* read out. The engine has no idea the brackets change the answer. This notebook is the
step before the voice.

Pipeline:

1. Parse the ASCII expression into a tree, hand-written recursive descent (no key needed).
2. Render the tree into one sentence, using an editable word table per language (no key needed).
3. Send that sentence to `text_to_speech.convert` and save a `.wav` into `outputs/` (needs a key).
4. Optionally hand long numbers to `text.transliterate` so they are spoken as words (needs a key).

**Two things to know before you read any further.**

**The words are this recipe's own choices, not any syllabus.** There is no single agreed way
to say school algebra in Hindi, Tamil or Telugu. This recipe picks one word for every symbol,
keeps them in one editable table per language, and claims nothing about any board or
curriculum. No textbook text ships here.

**Nothing here has been run.** There was no Sarvam API key on the machine this was built on,
so every cell below ships with empty output, nothing was spoken and nothing was heard. The
parser and the tables are fully tested offline; the two functions that call the API have never
met a live server. Run it yourself and listen.

In [ ]:
%pip install -r requirements.txt

## 1. The offline core

`equation_speech` sits next to this notebook. It imports nothing but the standard library, so
these cells run with no key, no network and not even `sarvamai` installed.

In [ ]:
from __future__ import annotations

import pathlib

import equation_speech as es

print("languages:", es.SUPPORTED_LANGUAGES)
print("speech model:", es.TTS_MODEL, "| character cap:", es.TTS_CHAR_CAP)
print("recipe folder:", pathlib.Path.cwd().name)

## 2. Parse, then render

`parse()` gives you the tree. `render()` turns a tree into a sentence. `verbalise()` does both
in one call.

Nothing is ever evaluated: `2+3` stays `2+3`. This reads maths aloud, it does not do maths.

In [ ]:
tree = es.parse("(a+b)^2")
print(tree)

for code_ in es.SUPPORTED_LANGUAGES:
    print(f"{code_}: {es.render(tree, code_)}")

## 3. The twelve worked examples

One table, all four languages. Watch the first two rows: `(a+b)^2` and `a+b^2` are the pair
this whole recipe exists to keep apart.

In [ ]:
for source in es.WORKED_EXAMPLES:
    print(source)
    for code_ in es.SUPPORTED_LANGUAGES:
        print(f"    {code_}  {es.verbalise(source, code_)}")
    print()

## 4. Why the parser is hand-written

The obvious shortcut for "I need an expression parser" is Python's own `ast.parse`. It would
have destroyed this recipe on the first example.

In Python, `^` is bitwise XOR and it binds **lower** than `+`. So `(a+b)^2` and `a+b^2` produce
structurally identical trees, and `2^3^2` comes out left-associative where a mathematical
exponent is right-associative. The one distinction this recipe makes is the one Python's parser
throws away, and it throws it away silently.

In [ ]:
import ast

print("ast.parse('(a+b)^2') ==", ast.dump(ast.parse("(a+b)^2", mode="eval").body))
print("ast.parse('a+b^2')   ==", ast.dump(ast.parse("a+b^2", mode="eval").body))
print("identical:", ast.dump(ast.parse("(a+b)^2", mode="eval").body)
      == ast.dump(ast.parse("a+b^2", mode="eval").body))

print()
print("this parser keeps them apart:", es.parse("(a+b)^2") != es.parse("a+b^2"))

## 5. Bracket words come from the structure, not from your parentheses

The renderer does not remember where you typed a bracket. It puts bracket words wherever the
sentence would otherwise be ambiguous by ear. So redundant brackets vanish, and brackets that
carry meaning turn into words.

These are the pairs that have to stay different. If any of them ever reads the same, the recipe
has failed at its one job.

In [ ]:
PAIRS = [
    ("(a+b)^2", "a+b^2"),
    ("a-(b-c)", "(a-b)-c"),
    ("(x^2)^3", "x^(2^3)"),
    ("-x^2", "(-x)^2"),
    ("sqrt(x)+1", "sqrt(x+1)"),
    ("3/4", "3.4"),
    ("3.40", "3.4"),
    ("x<=5", "x<5"),
]

for left, right in PAIRS:
    print(f"{left:>10}  ->  {es.verbalise(left, 'en-IN')}")
    print(f"{right:>10}  ->  {es.verbalise(right, 'en-IN')}")
    print()

print("redundant brackets are dropped:", es.verbalise("((((1))))", "en-IN"))

## 6. Numbers

Single digits become words from a ten-word table. Anything longer stays as digits and is left
for the voice, because naming an arbitrary integer in Hindi, Tamil or Telugu is a whole product
on its own. Five digits or more get comma-grouped, because the vendor documentation asks for
`'10,000'` rather than `'10000'` so the number is read as one number.

Decimals are read digit by digit after the point. The whole-number reading is equally valid and
is used by real people; digit-wise is a choice, not a correction, and both are named in the
module so nobody swaps them by accident.

In [ ]:
for source in ["0", "9", "34", "1234", "12000", "10000", "3.4", "3.40", "12.5", "3.14159"]:
    print(f"{source:>10}  en-IN: {es.verbalise(source, 'en-IN')}")
    print(f"{'':>10}  hi-IN: {es.verbalise(source, 'hi-IN')}")

print()
print("reading in use:", es.DECIMAL_READING)
print("the alternative:", es.DECIMAL_READING_ALTERNATIVE)

## 7. What it refuses, and where

Anything outside the grammar is an error carrying the index of the offending character. The ten
commonest near misses -- the characters people paste out of word processors and textbook PDFs
-- come back with the ASCII to type instead.

`x` is always a variable and never a multiplication sign, so `2 x 3` is a syntax error rather
than a guess. Guessing here would guess wrong in front of a student.

In [ ]:
BAD = ["2 x 3", "2(a+b)", "x²", "2 × 3", "x ≤ 5", "१+२",
       "1_0", "1e3", "3..4", "(a+b", "sqrt x"]

for source in BAD:
    try:
        es.parse(source)
    except es.ParseError as exc:
        print(f"{source!r:>12}  position {exc.position}: {exc.message}")

print()
print("near-miss table:", es.ASCII_SUGGESTIONS)

## 8. Editing the words

Every word lives in one `RuleTable` per language. There is no logic in them and no f-string
assembled anywhere in the renderer, so a teacher who disagrees with a word edits one string and
never opens the parser.

Note how the possessive particle is part of the stored phrase rather than something the code
adds. Hindi वर्ग is masculine and takes का; घात is feminine and takes की. Keeping the particle
inside the string means fixing an agreement error is a one-string edit, not a gender-agreement
engine.

In [ ]:
table = es.RULES["hi-IN"]
print("digits:        ", table.digits)
print("operators:     ", table.operators)
print("power words:   ", table.power_words)
print("brackets:      ", table.bracket_open, "/", table.bracket_close)
print("word order:    ", table.comparison_order)
print("variable words:", table.variable_words, "(empty by default)")

print()
print("x+y in hi-IN:", es.verbalise("x+y", "hi-IN"))

## 9. Speak it -- this cell needs an API key

Everything above ran offline. From here on you need a key in `.env`.

Three things this cell pins on purpose:

- The key is passed **explicitly**. The client's own default is read once, when the SDK is
  imported, so setting the environment variable afterwards is already too late.
- The parameter is `language_code`. The transliterate endpoint one module away calls the same
  idea `target_language_code`; mixing them up is a bug this repo has already fixed twice.
- `model` is passed on every call. The SDK omits it when you do not, leaving the server to
  choose, and the 2500-character cap belongs to `bulbul:v3` alone -- `bulbul:v2` stops at 1500.

One note on language codes if you extend this. The speech API accepts `od-IN` for Odia and does
not accept `or-IN`, even though this repo's rules file lists `or-IN`. That is issue #157. Do not
"correct" a recipe from the rules file.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running this cell."
    )

LANGUAGE = "hi-IN"
sentence = es.verbalise("(a+b)^2", LANGUAGE)
print("sentence:", sentence)
print("length:", len(sentence), "of", es.TTS_CHAR_CAP)

audio = es.speak(sentence, LANGUAGE, SARVAM_API_KEY)

output_dir = pathlib.Path("outputs")
output_dir.mkdir(exist_ok=True)
wav_path = output_dir / "equation.wav"
wav_path.write_bytes(audio)
print("wrote", wav_path, f"({len(audio)} bytes)")

## 10. Long numbers, spoken as words -- also needs a key

The one job the offline tables deliberately do not do. `text.transliterate` with
`spoken_form=True` and `spoken_form_numerals_language="native"` turns `34` into the target
language's word for thirty-four, server-side, so this recipe does not have to implement Indian
number naming.

Entirely optional. The sentence above is complete and speakable without it.

In [ ]:
spoken = es.spoken_numerals(es.verbalise("34", "hi-IN"), "hi-IN", SARVAM_API_KEY)
print("34 ->", spoken)

long_sentence = es.verbalise("x<=12000", "hi-IN")
print("before:", long_sentence)
print("after: ", es.spoken_numerals(long_sentence, "hi-IN", SARVAM_API_KEY))

## 11. What this does not do

Each of these is a parse error with a position, not a silent guess: matrices and vectors;
limits, summations and products; definite integrals with bounds; `sin`, `cos`, `log`, `ln`;
subscripts; multi-letter variables and Greek letters; implicit multiplication; scientific
notation; complex numbers; factorials; absolute value.

And out of the recipe entirely: naming integers beyond 0 to 9 offline; languages beyond the
four; evaluating the expression; LaTeX or MathML input; speech back to an equation; and
splitting an over-long sentence across two audio files, which is a worse answer than telling
the caller. A sentence over the cap raises `SpeechLengthError` before any client is built.

If you speak Hindi, Tamil or Telugu, the word tables in `equation_speech.py` are the first
place to look. They have not been reviewed by anybody who does.